# День 2. Сравнение OCR-движков и предобработки

Запускает `scripts/benchmark_ocr.py` в Colab: **Tesseract vs EasyOCR**, с предобработкой и без неё, на чистых документах и на «плохих снимках» (наклон, размытие, низкое разрешение, тень, блик, JPEG-шум).
Метрики считает `evaluate.py` организаторов, поэтому цифры сопоставимы с итоговой автопроверкой.

**Перед запуском:** `Среда выполнения -> Сменить тип среды -> T4 GPU` (ускоряет EasyOCR). Tesseract работает на CPU в любом случае.

## 1. Системные пакеты и код проекта

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-rus > /dev/null
!tesseract --list-langs

In [ ]:
# Вариант A: клонировать репозиторий. Для приватного репозитория вставьте токен вместо <TOKEN>
# (GitHub -> Settings -> Developer settings -> Personal access tokens, право repo).
REPO_URL = "https://github.com/<ЛОГИН>/<РЕПО>.git"   # <- подставьте свой
import os, subprocess
if not os.path.exists("/content/project"):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], capture_output=True, text=True)
    print(r.stdout, r.stderr)
%cd /content/project
!ls

Если клонирование не удалось (приватный репозиторий без токена), используйте **вариант B**: упакуйте проект в zip (без `.venv` и `data/open`) и загрузите:

```python
from google.colab import files; import zipfile
up = files.upload()                       # выберите project.zip
zipfile.ZipFile(list(up)[0]).extractall("/content/project")
%cd /content/project
```

## 2. Зависимости

In [ ]:
!pip -q install pytesseract pymupdf pyyaml "pydantic>=2" easyocr
import cv2, numpy, torch
print("opencv", cv2.__version__, "| numpy", numpy.__version__, "| GPU:", torch.cuda.is_available())

## 3. Документы организаторов

Загрузите архив с PNG (`documents/open-001.png` ... `open-030.png`). Создайте его на своём компьютере:

```powershell
Compress-Archive -Path data\open\documents -DestinationPath documents.zip
```

In [ ]:
from google.colab import files; import zipfile, os
up = files.upload()                       # выберите documents.zip
os.makedirs("data/open", exist_ok=True)
zipfile.ZipFile(list(up)[0]).extractall("data/open")
!ls data/open/documents | head -3
!ls data/open/documents | wc -l

## 4. Быстрая проверка (1-2 минуты)

In [ ]:
!PYTHONPATH=src python -m pytest -q tests/test_oracle.py tests/test_degrade.py
!PYTHONPATH=src python scripts/benchmark_ocr.py --docs-dir data/open --engines tesseract,easyocr --conditions clean,photo --limit 1 --out outputs/smoke

## 5. Полный бенчмарк

Занимает заметное время (десятки минут: 10 условий x 2 движка x 2 режима предобработки x 30 документов).
Если хочется быстрее: `--limit 3` (по 3 документа каждого типа) или меньше условий через `--conditions clean,rot15,photo`.

In [ ]:
!PYTHONPATH=src python scripts/benchmark_ocr.py --docs-dir data/open --engines tesseract,easyocr --out outputs/bench

## 6. Таблица результатов

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/bench/bench_table.csv")
pd.set_option("display.width", 200)
df

In [ ]:
# Сводка: точность полей по условиям (строки) и конфигурациям (столбцы)
df["config"] = df["engine"] + " / pre=" + df["preprocess"]
pivot = df.pivot(index="condition", columns="config", values="field_acc")
pivot.loc[df["condition"].unique()].round(3).style.background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0)

In [ ]:
# Скорость: секунды на документ (OCR / всего)
df.pivot(index="condition", columns="config", values="total_sec_per_doc").round(1)

## 7. Забрать результаты

Скачайте `bench_table.md` и `bench_results.json` и положите в репозиторий: `docs/ocr_benchmark.md`.
Пришлите таблицу ассистенту: по ней выбираем движок и стратегию (запасной движок на сложные снимки).

In [ ]:
files.download("outputs/bench/bench_table.md")
files.download("outputs/bench/bench_results.json")